# Домашняя работа 4: память, собственный компонент и ветвление

Продолжение личного ассистента из задания 2: сохранение города, погода, акции и маршрутизация запросов. Для демонстрации используется локальная Maple-Preview.

In [1]:
import json
import tempfile
from pathlib import Path

from src.agent import ToolAgent
from src.config import LOCAL_MODELS
from src.file_fact_memory import FileFactMemory
from src.providers.maple_server_provider import MapleServerProvider

## 1. Инициализация

Файловая память хранит факты пользователя. `ConversationBufferWindowMemory` содержит последние два обмена, а `ConversationSummaryMemory` — сводку старых. Данные погоды и цен в этом проекте учебные; источник новостей пока не подключён.

In [2]:
demo_dir = tempfile.TemporaryDirectory()
memory_file = str(Path(demo_dir.name) / "memory.json")
memory = FileFactMemory(file_path=memory_file, session_id="student", show_service_cell=False)
provider = MapleServerProvider(model_path=LOCAL_MODELS["maple"]["path"], model_alias="maple")
agent = ToolAgent(provider=provider, memory=memory, show_cell=False, window_size=2)
print(agent.get_greeting())
print("Память окна:", type(agent.dialogue_memory.window_memory).__name__)
print("Память сводки:", type(agent.dialogue_memory.summary_memory).__name__)

C:\Users\veles\AppData\Local\Programs\Python\Python312\Lib\site-packages\pydantic\main.py:263: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


Привет! Я твой личный ассистент. Я запоминаю важные факты о тебе в файл memory.json, умею узнавать цены криптовалют (BTC, ETH), акций российских компаний (Сбер, Газпром, Яндекс) и помогать с задачами. Как к тебе обращаться?
Память окна: ConversationBufferWindowMemory
Память сводки: ConversationSummaryMemory


C:\Users\veles\PycharmProjects\agents_lab1\src\dialogue_memory.py:19: LangChainDeprecationWarning: The class `ConversationBufferWindowMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  self.window_memory = ConversationBufferWindowMemory(
C:\Users\veles\PycharmProjects\agents_lab1\src\dialogue_memory.py:22: LangChainDeprecationWarning: The class `ConversationSummaryMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.

## 2. Окно диалога и суммаризация

Указываем город пользователя и спрашиваем погоду. После третьего обмена первый попадает в суммаризирующую память, а два последних остаются в окне.

In [3]:
messages = [
    "Меня зовут Алексей.",
    "Запомни мой город: Екатеринбург",
    "Какая погода в моём городе?",
]
for message in messages:
    result = agent.run(message)
    print(f"Пользователь: {message}")
    print(f"Ассистент: {result['final_answer']}\n")

assert result["request_type"] == "weather"
assert result["steps"][0]["action"] == "get_weather"
assert result["steps"][0]["args"]["city"] == "Екатеринбург"
state = agent.dialogue_memory.load_memory_variables()
print("Сводка:", state["summary"])
print("Обменов в окне:", len(state["window"]))
for turn in state["window"]:
    print("-", turn["user"])
assert state["summary"] and len(state["window"]) == 2

Пользователь: Меня зовут Алексей.
Ассистент: Привет, Алексей!

Пользователь: Запомни мой город: Екатеринбург
Ассистент: Город сохранён: Екатеринбург



Пользователь: Какая погода в моём городе?
Ассистент: Погода в городе Екатеринбург: +9°C, ясно, сухо (демонстрационные данные)

Сводка: Вот обновлённая сводка:

- **Имя:** Алексей (упомянут в обмене Human: Меня зовут Алексей).
- **Приветствие:** AI: Привет, Алексей! (упомянуто в обмене AI: Привет, Алексей!).
- **Цель:** (не указана ранее, не добавлено).
- **Важные факты:** обмен включает приветствие с именем Алексей.
Обменов в окне: 2
- Запомни мой город: Екатеринбург
- Какая погода в моём городе?


## 3. Собственный компонент: сохранение города

`SaveUserCity` — инструмент LangChain с Pydantic-схемой. Он проверяет название города и записывает его в файл памяти. Сессии пользователя хранятся отдельно.

In [4]:
print(agent.city_tool.invoke({"city": "Казань"}))
restored = FileFactMemory(file_path=memory_file, session_id="student", show_service_cell=False)
other = FileFactMemory(file_path=memory_file, session_id="another_student", show_service_cell=False)
print("Город после повторного чтения:", restored.facts.get("city"))
print("Факты другой сессии:", other.facts)
try:
    agent.city_tool.invoke({"city": "123"})
except Exception:
    print("Некорректное название города отклонено")
assert restored.facts["city"] == "Казань" and not other.facts

Город сохранён: Казань
Город после повторного чтения: Казань
Факты другой сессии: {}
Некорректное название города отклонено


## 4. Ветвление по типу запроса

`RunnableBranch` выбирает обработку погоды, финансов, новостей или общего вопроса. В этом учебном проекте погода и цены заданы константами. Для новостей требуется подключить источник; агент не придумывает актуальные новости.

In [5]:
for request_type in ["weather", "finance", "news", "general"]:
    guidance = agent.request_branch.invoke({"request_type": request_type, "city": "Казань"})
    print(request_type, "→", guidance)

result = agent.run("Какая цена акций Сбербанка?")
print("\nКатегория:", result["request_type"])
print("Ответ:", result["final_answer"])
assert result["request_type"] == "finance"
assert result["steps"][0]["action"] == "get_stock_price"
assert "демонстрационные данные" in result["final_answer"]

result = agent.run("Какие сегодня новости?")
print("\nКатегория:", result["request_type"])
print("Ответ:", result["final_answer"])
assert result["request_type"] == "news" and "не подключён" in result["final_answer"]

weather → Запрос о погоде. Используй get_weather и город: Казань.
finance → Запрос о финансах. Используй подходящий инструмент для акций, криптовалюты или курса.
news → Для новостей нужен подключённый источник.
general → Ответь на запрос, используя сохранённые факты, если они нужны.



Категория: finance
Ответ: Акции Сбербанка (SBER): 312.40 RUB (демонстрационные данные)



Категория: news
Ответ: Источник новостей пока не подключён, поэтому получить актуальные новости я не могу.


## 5. Завершение

Для чата запустите `python main.py --model maple --session student`. Сохранённый город и другие факты остаются в файле памяти; окно и сводка относятся к текущему запуску.

In [6]:
provider.close()
demo_dir.cleanup()
print("Локальный сервер остановлен, временные данные удалены.")

Локальный сервер остановлен, временные данные удалены.
